# FTW Baseline — Result Visualization

Displays training/test results for the 3-class U-Net + EfficientNet-B3 baseline.

**Panels shown per sample:**
1. RGB composite (Sentinel-2 bands 0,1,2 — window A)
2. Ground truth mask (background / crop / boundary)
3. Predicted mask
4. Vectorized field polygons overlaid on RGB

**Prerequisites:**
- Run `ftw model test` to produce the metrics CSV (`outputs/test_metrics.csv`)
- Run `ftw inference run` + `ftw inference polygonize` to produce `outputs/austria_pred.tif` and `outputs/austria_fields.parquet`
- Austria test data available under `data/ftw/austria/`

In [ ]:
import glob
import random

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import ListedColormap
from rasterio.plot import show

random.seed(42)

## 1. Metrics Summary

In [ ]:
METRICS_CSV = "../outputs/test_metrics.csv"

try:
    df = pd.read_csv(METRICS_CSV)
    display(df.T)  # Transpose for readability
except FileNotFoundError:
    print(f"Metrics file not found: {METRICS_CSV}")
    print("Run: ftw model test --model <checkpoint.ckpt> --countries austria --model_predicts_3_classes --test_on_3_classes --out outputs/test_metrics.csv")

## 2. Sample Patch Predictions

Loads individual 256×256 Austria test patches directly from the FTW dataset and shows
RGB | Ground Truth | Prediction side-by-side. Requires a trained checkpoint.

In [ ]:
import torch
from ftw_tools.training.datasets import FTW
from ftw_tools.training.trainers import CustomSemanticSegmentationTask

CHECKPOINT = "../logs/FTW-Release-Full-3-class/lightning_logs/version_0/checkpoints/last.ckpt"
DATA_ROOT = "../data/ftw"
N_SAMPLES = 6

# Class colormap: 0=background (grey), 1=crop (green), 2=boundary (red), 3=unknown (black)
CMAP = ListedColormap(["#888888", "#4CAF50", "#F44336", "#000000"])
CLASS_LABELS = ["background", "crop", "boundary", "unknown"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
task = CustomSemanticSegmentationTask.load_from_checkpoint(CHECKPOINT, map_location="cpu")
model = task.model.eval().to(device)
print(f"Model loaded on {device}")

In [ ]:
ds = FTW(root=DATA_ROOT, countries=["austria"], split="test", load_boundaries=True)
indices = random.sample(range(len(ds)), min(N_SAMPLES, len(ds)))

fig, axes = plt.subplots(len(indices), 4, figsize=(18, 4 * len(indices)))
if len(indices) == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(indices):
    sample = ds[idx]
    img = sample["image"]  # (C, H, W) float tensor, already normalized
    mask = sample["mask"].squeeze(0).numpy()  # (H, W)

    # Model prediction
    with torch.inference_mode():
        logits = model(img.unsqueeze(0).to(device))  # (1, num_classes, H, W)
    pred = logits.argmax(dim=1).squeeze(0).cpu().numpy()  # (H, W)

    # RGB from first 3 channels (window A: B, G, R or R, G, B depending on order)
    rgb = img[:3].permute(1, 2, 0).numpy()
    rgb = np.clip((rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8), 0, 1)

    # Panel 0: RGB
    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f"RGB (sample {idx})")
    axes[row, 0].axis("off")

    # Panel 1: Ground Truth
    axes[row, 1].imshow(mask, cmap=CMAP, vmin=0, vmax=3, interpolation="nearest")
    axes[row, 1].set_title("Ground Truth")
    axes[row, 1].axis("off")

    # Panel 2: Prediction
    axes[row, 2].imshow(pred, cmap=CMAP, vmin=0, vmax=3, interpolation="nearest")
    axes[row, 2].set_title("Prediction")
    axes[row, 2].axis("off")

    # Panel 3: Overlay (prediction as transparent mask on RGB)
    axes[row, 3].imshow(rgb)
    overlay = np.zeros((*pred.shape, 4), dtype=np.float32)  # RGBA
    overlay[pred == 1] = [0.3, 0.8, 0.3, 0.4]   # crop: semi-transparent green
    overlay[pred == 2] = [0.9, 0.2, 0.2, 0.7]   # boundary: semi-transparent red
    axes[row, 3].imshow(overlay)
    axes[row, 3].set_title("Prediction Overlay")
    axes[row, 3].axis("off")

# Legend
legend_patches = [
    mpatches.Patch(color="#888888", label="Background"),
    mpatches.Patch(color="#4CAF50", label="Crop"),
    mpatches.Patch(color="#F44336", label="Boundary"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=12)
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("../outputs/sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to outputs/sample_predictions.png")

## 3. Full-Scene Inference Output

Visualizes the prediction TIF produced by `ftw inference run` and the polygon
boundaries produced by `ftw inference polygonize`.

In [ ]:
PRED_TIF = "../outputs/austria_pred.tif"
FIELDS_PARQUET = "../outputs/austria_fields.parquet"

try:
    with rasterio.open(PRED_TIF) as src:
        pred_raster = src.read(1)  # single-band class prediction
        transform = src.transform
        crs = src.crs
    print(f"Prediction raster: {pred_raster.shape}, CRS: {crs}")

    fields = gpd.read_parquet(FIELDS_PARQUET)
    print(f"Polygons: {len(fields)} fields, CRS: {fields.crs}")

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Left: prediction raster
    im = axes[0].imshow(pred_raster, cmap=CMAP, vmin=0, vmax=3, interpolation="nearest")
    axes[0].set_title("Prediction Raster (full scene)")
    axes[0].axis("off")
    plt.colorbar(im, ax=axes[0], fraction=0.03)

    # Right: vector polygons
    fields.plot(ax=axes[1], facecolor="#4CAF5066", edgecolor="#F44336", linewidth=0.5)
    axes[1].set_title(f"Vectorized Field Polygons (n={len(fields)})")
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig("../outputs/full_scene_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to outputs/full_scene_results.png")

except FileNotFoundError as e:
    print(f"File not found: {e}")
    print("Run inference first:")
    print("  ftw inference run --model <checkpoint.ckpt> --input <scene.tif> --output outputs/austria_pred.tif")
    print("  ftw inference polygonize outputs/austria_pred.tif --output outputs/austria_fields.parquet --simplify 15 --min_size 500")

## 4. Polygon Size Distribution

In [ ]:
try:
    fields_utm = fields.to_crs(fields.estimate_utm_crs())
    areas_ha = fields_utm.geometry.area / 10_000  # sq m → hectares

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(areas_ha, bins=50, color="#4CAF50", edgecolor="white")
    axes[0].set_xlabel("Field area (ha)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Field Size Distribution")

    axes[1].hist(areas_ha[areas_ha < 20], bins=50, color="#2196F3", edgecolor="white")
    axes[1].set_xlabel("Field area (ha) — clipped at 20 ha")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Field Size Distribution (< 20 ha)")

    plt.tight_layout()
    plt.show()

    print(f"Total fields: {len(fields_utm)}")
    print(f"Median area:  {areas_ha.median():.2f} ha")
    print(f"Mean area:    {areas_ha.mean():.2f} ha")
    print(f"Max area:     {areas_ha.max():.2f} ha")
except Exception as e:
    print(f"Skipped polygon analysis: {e}")